# Data Acquisition and Vectorization

This notebook demonstrates how to:
1. Connect to Confluence using REST API and retrieve project documentation
2. Process and chunk the data
3. Generate embeddings and store in vector database

In [ ]:
import sys
sys.path.append('..')

from src.config import ConfigConfluenceRag
from src.confluence.rest_client import ConfluenceRestClient
from src.confluence.parser import ConfluenceParser
from src.rag.vectorstore import VectorStore
from src.rag.embeddings import EmbeddingManager
from loguru import logger
import pandas as pd

## 1. Validate Configuration

In [ ]:
if not ConfigConfluenceRag.validate():
    raise ValueError("Configuration validation failed. Please check your .env file.")

print("✅ Configuration validated successfully")
print(f"Confluence URL: {ConfigConfluenceRag.CONFLUENCE_URL}")
print(f"Confluence Username: {ConfigConfluenceRag.CONFLUENCE_USERNAME}")
print(f"Confluence Space: {ConfigConfluenceRag.CONFLUENCE_SPACE_KEY}")
print(f"Embedding Model: {ConfigConfluenceRag.EMBEDDING_MODEL}")
print(f"Vector DB Path: {ConfigConfluenceRag.VECTOR_DB_PATH}")

## 2. Retrieve Confluence Pages

In [ ]:
# Initialize Confluence REST API client
print("Initializing Confluence REST API client...")
print(f"URL: {ConfigConfluenceRag.CONFLUENCE_URL}")
print(f"Username: {ConfigConfluenceRag.CONFLUENCE_USERNAME}")
print(f"Space: {ConfigConfluenceRag.CONFLUENCE_SPACE_KEY}")
print()

# Try Bearer token authentication first (common for on-premise Confluence)
print("Attempting Bearer token authentication...")
confluence_client = ConfluenceRestClient(
    base_url=ConfigConfluenceRag.CONFLUENCE_URL,
    username=ConfigConfluenceRag.CONFLUENCE_USERNAME,
    api_token=ConfigConfluenceRag.CONFLUENCE_API_TOKEN,
    verify_ssl=False,
    auth_type='bearer'
)

if not confluence_client.test_connection():
    print("\nBearer token failed. Trying Basic authentication...")
    confluence_client = ConfluenceRestClient(
        base_url=ConfigConfluenceRag.CONFLUENCE_URL,
        username=ConfigConfluenceRag.CONFLUENCE_USERNAME,
        api_token=ConfigConfluenceRag.CONFLUENCE_API_TOKEN,
        verify_ssl=False,
        auth_type='basic'
    )
    if not confluence_client.test_connection():
        raise Exception("❌ Authentication failed with both Bearer and Basic auth methods")

print("\n✓ Authentication successful!")
print()

# Fetch all pages from the space
print("Retrieving Confluence pages from REST API...")
pages = confluence_client.get_all_pages_from_space(ConfigConfluenceRag.CONFLUENCE_SPACE_KEY)
print(f"\n✓ Retrieved {len(pages)} pages from Confluence")

In [ ]:
# Display sample page information
if pages:
    sample_page = pages[0]
    print(f"Sample Page Title: {sample_page.title}")
    print(f"URL: {sample_page.url}")
    print(f"Space: {sample_page.space_key} - {sample_page.space_name}")
    print(f"Author: {sample_page.author}")
    print(f"Version: {sample_page.version}")
    print(f"Modified: {sample_page.modified_date}")
    print(f"Content HTML size: {len(sample_page.content_html or '')} characters")
    print(f"Content Text size: {len(sample_page.content_text or '')} characters")
    print(f"External links: {len(sample_page.external_links)} links")
    if sample_page.parent_title:
        print(f"Parent: {sample_page.parent_title} (ID: {sample_page.parent_id})")
    print(f"Ancestors: {len(sample_page.ancestors)} level(s)")
    print(f"Children: {len(sample_page.children)} page(s)")

## 2a. Save Raw Pages to JSON (Before Vectorization)

In [ ]:
# Save raw Confluence pages to JSON for backup and future reference
# This preserves all the data before vectorization
from pathlib import Path

json_output_path = Path('Data_Storage/confluence_pages.json')
json_output_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Saving raw pages to: {json_output_path}")
confluence_client.save_pages_to_json(pages, str(json_output_path))
print(f"✅ Saved {len(pages)} pages to JSON")
print(f"   File size: {json_output_path.stat().st_size / 1024 / 1024:.2f} MB")

## 3. Convert Pages to Processing Format

In [ ]:
# Convert ConfluencePage objects to format for processing
# The REST API client already extracts plain text, so we'll use that
print("Converting pages to processing format...")

pages_content = []
for page in pages:
    # Use the already-extracted plain text from REST API
    pages_content.append({
        'id': page.id,
        'title': page.title,
        'space': page.space_key,
        'url': page.url,
        'content_html': page.content_html or '',
        'text': page.content_text or '',  # Already extracted by REST API
        'external_links': page.external_links,
        'author': page.author,
        'version': page.version,
        'modified_date': page.modified_date,
        'parent_id': page.parent_id,
        'parent_title': page.parent_title,
        'ancestors': page.ancestors,
        'children': page.children
    })

print(f"Converted {len(pages_content)} pages successfully")

In [ ]:
# Display page statistics
df_pages = pd.DataFrame([
    {
        'title': page['title'],
        'text_length': len(page['text']),
        'num_external_links': len(page['external_links']),
        'has_parent': page['parent_id'] is not None,
        'num_ancestors': len(page['ancestors']),
        'num_children': len(page['children'])
    }
    for page in pages_content
])

print("\nPage Statistics:")
print(df_pages.describe())
print()
df_pages.head(10)

## 4. Chunk Documents for Vectorization

In [ ]:
# Initialize parser for chunking
confluence_parser = ConfluenceParser()

# Chunk Confluence pages
confluence_chunks = []

for page in pages_content:
    # Use the text already extracted by the REST API
    text_chunks = confluence_parser.chunk_text(
        page['text'],
        chunk_size=ConfigConfluenceRag.CHUNK_SIZE,
        chunk_overlap=ConfigConfluenceRag.CHUNK_OVERLAP
    )
    
    for i, chunk in enumerate(text_chunks):
        confluence_chunks.append({
            'text': chunk,
            'metadata': {
                'title': page['title'],
                'url': page['url'],
                'source_type': 'confluence',
                'chunk_id': i,
                'space': page['space'],
                'author': page['author'],
                'version': page['version'],
                'page_id': page['id']
            }
        })

print(f"Created {len(confluence_chunks)} chunks from {len(pages_content)} Confluence pages")

## 5. Generate Embeddings and Store in Vector Database

In [ ]:
# Initialize embedding manager
embedding_manager = EmbeddingManager(model_name=ConfigConfluenceRag.EMBEDDING_MODEL)
print(f"Embedding model: {embedding_manager.model_name}")
print(f"Embedding dimension: {embedding_manager.embedding_dimension}")

In [ ]:
# Initialize vector store
vector_store = VectorStore(
    persist_directory=ConfigConfluenceRag.VECTOR_DB_PATH,
    collection_name='confluence_docs'
)

# Clear existing data (optional)
if vector_store.count() > 0:
    print(f"Clearing existing {vector_store.count()} documents from vector store...")
    vector_store.clear_collection()
    print("Vector store cleared")

In [ ]:
# Prepare data for storage
texts = [chunk['text'] for chunk in confluence_chunks]
metadatas = [chunk['metadata'] for chunk in confluence_chunks]
ids = [f"doc_{i}" for i in range(len(confluence_chunks))]

# Add documents to vector store
print("\nAdding documents to vector store...")
vector_store.add_documents(texts=texts, metadatas=metadatas, ids=ids)

print(f"✅ Successfully added {vector_store.count()} documents to vector store")

## 6. Verify Vector Store

In [ ]:
# Test query
test_query = "What data science projects are documented?"
print(f"Test query: {test_query}\n")

results = vector_store.query(query_text=test_query, n_results=3)

print("Top 3 results:")
for i, (doc, meta, dist) in enumerate(zip(results['documents'], results['metadatas'], results['distances']), 1):
    print(f"\n{i}. Title: {meta['title']}")
    print(f"   Source: {meta['source_type']}")
    print(f"   Distance: {dist:.4f}")
    print(f"   Preview: {doc[:200]}...")

## Summary

This notebook has:
- ✅ Retrieved and parsed Confluence pages using REST API
- ✅ Converted pages to processing format with metadata
- ✅ Chunked documents for optimal retrieval
- ✅ Generated embeddings using sentence transformers
- ✅ Stored documents in ChromaDB vector database

The vector database is now ready for use with the RAG pipeline!